In [1]:
from pathlib import Path
import os
# The project folder is the folder that contains data/, notebooks/, results/ ...
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROJECT_DIR = Path(os.environ.get('PROJECT_DIR', PROJECT_DIR))
print('Project folder:', PROJECT_DIR)
assert (PROJECT_DIR / 'data').exists(), 'Project folder not found - open this notebook from the notebooks folder of the project (see your Steps file)'

Project folder: C:\Users\Reach\Documents\game-chat-toxicity-ml


In [2]:
# gensim is installed by requirements.txt. If it is missing, run in a cell:  %pip install gensim
import re, html, json
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
import gensim.downloader as api
from collections import Counter

USE_SAMPLE = False
SEED = 42
LAB_DIR, EMB_DIR = PROJECT_DIR / 'data' / 'labelled', PROJECT_DIR / 'models' / 'emb'
CORPUS = PROJECT_DIR / 'data' / 'processed' / 'unlabelled_corpus.csv'
EMB_DIR.mkdir(parents=True, exist_ok=True)

prefix = 'sample_' if USE_SAMPLE else ''
train = pd.read_csv(LAB_DIR / f'{prefix}train.csv')
print('train messages:', len(train))

train messages: 1583


In [3]:
def clean_text(s):
    s = html.unescape(str(s)).lower()
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"[\u2019'`]", "", s)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)
    s = re.sub(r"[^a-z0-9?! ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def tokenize(s):
    return re.findall(r"[a-z0-9]+|[?!]", clean_text(s))

MAX_LEN = 30
min_count = 1 if USE_SAMPLE else 2
cnt = Counter(w for s in train.text for w in tokenize(s))
words = sorted([w for w, c in cnt.items() if c >= min_count])
vocab = {'<pad>': 0, '<unk>': 1}
for w in words: vocab[w] = len(vocab)
print('vocabulary size:', len(vocab), '| most common:', cnt.most_common(12))

lens = train.text.map(lambda s: len(tokenize(s)))
print('messages longer than MAX_LEN=%d: %.1f%%' % (MAX_LEN, 100 * (lens > MAX_LEN).mean()))
json.dump({'vocab': vocab, 'max_len': MAX_LEN}, open(EMB_DIR / 'vocab.json', 'w'))

vocabulary size: 713 | most common: [('i', 175), ('you', 152), ('u', 138), ('?', 132), ('and', 113), ('report', 107), ('a', 82), ('fucking', 79), ('is', 73), ('to', 73), ('ez', 64), ('the', 63)]
messages longer than MAX_LEN=30: 0.0%


In [4]:
if CORPUS.exists() and not USE_SAMPLE:
    corpus = pd.read_csv(CORPUS, encoding='utf-8').text.astype(str)
else:
    print('(no unlabelled corpus used - training on the labelled train messages only)')
    corpus = train.text
sentences = [tokenize(s) for s in corpus]
sentences = [s for s in sentences if len(s) >= 2]
print('word2vec training sentences:', len(sentences))

w2v = {}
for dim in (50, 100):
    w2v[dim] = Word2Vec(sentences, vector_size=dim, window=5, min_count=2 if USE_SAMPLE else 3,
                        workers=2, epochs=10, seed=SEED).wv
    print(f'word2vec {dim}-dim trained, vocabulary: {len(w2v[dim].key_to_index)}')

word2vec training sentences: 196007
word2vec 50-dim trained, vocabulary: 8295
word2vec 100-dim trained, vocabulary: 8295


In [5]:
glove = {50: api.load('glove-wiki-gigaword-50'), 100: api.load('glove-wiki-gigaword-100')}
print({d: len(g.key_to_index) for d, g in glove.items()}, 'words each')

{50: 400000, 100: 400000} words each


In [6]:
def similar(model, w, n=5):
    return [x for x, _ in model.most_similar(w, topn=n)] if w in model.key_to_index else '(unknown word)'

for w in ['noob', 'feeder', 'gg', 'report', 'trash']:
    print(f'{w:8s} word2vec-100 (our chat): {similar(w2v[100], w)}')
    print(f'{"":8s} GloVe-100 (news text)  : {similar(glove[100], w)}')

noob     word2vec-100 (our chat): ['retard', 'idiot', 'tard', 'pro', 'nob']
         GloVe-100 (news text)  : ['saibot', 'εἰς', 'usanee', 'tom.fowler@chron.com', 'bobolink']
feeder   word2vec-100 (our chat): ['flamer', 'nob', 'instalocker', 'nab', 'unskilled']
         GloVe-100 (news text)  : ['hogs', 'bellies', '0.375', 'feeders', 'cattle']
gg       word2vec-100 (our chat): ['bg', 'ggwp', 'wp', 'pleas', 'gj']
         GloVe-100 (news text)  : ['allin', '3,3', 'kl', 'fh', 'jfs']
report   word2vec-100 (our chat): ['reprot', 'rpeort', 'honor', 'rep', 'reported']
         GloVe-100 (news text)  : ['reports', 'reported', 'cited', 'citing', 'according']
trash    word2vec-100 (our chat): ['dog', 'russian', 'retarded', 'toxic', 'moron']
         GloVe-100 (news text)  : ['garbage', 'rubbish', 'dump', 'pile', 'waste']


In [7]:
tokens_total = sum(cnt[w] for w in words)
coverage = {}
for name, models in (('w2v', w2v), ('glove', glove)):
    for dim, kv in models.items():
        mat = np.zeros((len(vocab), dim), dtype='float32')
        found = [w for w in words if w in kv.key_to_index]
        for w in found: mat[vocab[w]] = kv[w]
        np.save(EMB_DIR / f'{name}_{dim}.npy', mat)
        coverage[f'{name}_{dim}'] = {'vocab_words_found_pct': round(100 * len(found) / len(words), 1),
                                     'word_occurrences_found_pct': round(100 * sum(cnt[w] for w in found) / tokens_total, 1)}
cov = pd.DataFrame(coverage).T
display(cov)
if not USE_SAMPLE:
    json.dump(coverage, open(EMB_DIR / 'coverage.json', 'w'), indent=2)

,vocab_words_found_pct,word_occurrences_found_pct
w2v_50,99.6,99.9
w2v_100,99.6,99.9
glove_50,93.7,97.8
glove_100,93.7,97.8
